## Deep Research Agent
Agentic pipeline: Clarify -> Plan -> Search (parallel) -> Sufficiency check -> Write -> Evaluate -> Email

- Manager is a true Agent with sub-agents exposed as tools and a handoff to the email agent
- Clarifier auto-generates 3 scoping questions before any search
- Sufficiency agent triggers extra search rounds when coverage is thin (up to 2 retries)
- Evaluator scores the report and requests revision if score is below 7
- DuckDuckGo search replaces WebSearchTool so any provider works without an OpenAI key
- Input guardrail blocks harmful queries; output guardrail checks report quality
- Provider switchable via RESEARCH_PROVIDER env var: openrouter | openai | groq

In [ ]:
!uv pip install openai-agents sendgrid gradio python-dotenv ddgs

In [ ]:
from __future__ import annotations
import asyncio
import json
import os

import sendgrid
from ddgs import DDGS
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel, Field
from sendgrid.helpers.mail import Content, Email, Mail, To

import gradio as gr
from agents import (
    Agent, GuardrailFunctionOutput, ModelSettings, OpenAIChatCompletionsModel,
    Runner, RunContextWrapper, function_tool, gen_trace_id, input_guardrail,
    set_default_openai_api, set_default_openai_client, set_tracing_disabled,
    set_tracing_export_api_key, trace,
)
from agents.exceptions import InputGuardrailTripwireTriggered

In [ ]:
load_dotenv(override=True)

# Change RESEARCH_PROVIDER to switch provider: openrouter | openai | groq
# Note: Groq does not support json_schema structured outputs -- use openrouter or openai
PROVIDER = os.environ.get("RESEARCH_PROVIDER", "openrouter")

_PROVIDERS = {
    "cerebras":    {"model": "qwen-3-235b-a22b-instruct-2507",    "base_url": "https://api.cerebras.ai/v1",         "api_key_env": "CEREBRAS_API_KEY"},
    "openrouter":  {"model": "meta-llama/llama-3.3-70b-instruct", "base_url": "https://openrouter.ai/api/v1",       "api_key_env": "OPENROUTER_API_KEY"},
    "huggingface": {"model": "openai/gpt-oss-120b",               "base_url": "https://router.huggingface.co/v1",   "api_key_env": "HF_TOKEN"},
    "groq":        {"model": "llama-3.3-70b-versatile",           "base_url": "https://api.groq.com/openai/v1",     "api_key_env": "GROQ_API_KEY"},
    "openai":      {"model": "gpt-4o-mini",                       "base_url": "https://api.openai.com/v1",          "api_key_env": "OPENAI_API_KEY"},
}
_cfg = _PROVIDERS.get(PROVIDER, _PROVIDERS["openrouter"])
_client = AsyncOpenAI(api_key=os.environ.get(_cfg["api_key_env"], ""), base_url=_cfg["base_url"])

set_default_openai_api("chat_completions")
set_default_openai_client(_client)

# OpenAIChatCompletionsModel bypasses the SDK prefix router
# which rejects model names containing a slash (e.g. meta-llama/...)
DEFAULT_MODEL = OpenAIChatCompletionsModel(model=_cfg["model"], openai_client=_client)

# When using a non-OpenAI provider, the SDK still needs a valid OpenAI API key
# to export traces to platform.openai.com/traces.
# set_tracing_export_api_key separates the inference key from the tracing key,
# so you get full trace visibility even when running on OpenRouter, Groq, etc.
_openai_key = os.environ.get("OPENAI_API_KEY", "")
if _openai_key:
    set_tracing_export_api_key(_openai_key)
else:
    set_tracing_disabled(True)

# TEST_MODE: set True for a dry run before recording.
# Cuts all token limits to the minimum and disables retry loops so the full
# pipeline completes quickly without burning through your rate limit allowance.
# Set back to False for the real demo.
TEST_MODE = False

print(f"Provider: {PROVIDER} | Model: {_cfg['model']} | Test mode: {TEST_MODE}")

In [ ]:
class ClarifyingQuestions(BaseModel):
    questions:       list[str] = Field(description="Three clarifying questions to focus the research.")
    context_summary: str       = Field(description="One-sentence summary of the query.")

class WebSearchItem(BaseModel):
    query:    str = Field(description="The search term.")
    reason:   str = Field(description="Why this search is relevant.")
    priority: int = Field(description="Priority 1 (critical) to 3 (nice-to-have).", ge=1, le=3)

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="Ordered list of web searches.")

class ResearchSufficiencyCheck(BaseModel):
    is_sufficient:     bool       = Field(description="True if current results are enough for a thorough report.")
    missing_aspects:   list[str]  = Field(description="Topics still not covered.")
    additional_queries: list[str] = Field(description="New queries to fill the gaps.")
    reasoning:         str        = Field(description="Brief explanation.")

class ReportData(BaseModel):
    short_summary:       str       = Field(description="2-3 sentence executive summary.")
    markdown_report:     str       = Field(description="Full detailed report in Markdown (min 1000 words).")
    follow_up_questions: list[str] = Field(description="Suggested directions for further research.")

class ReportEvaluation(BaseModel):
    is_approved:             bool      = Field(description="True if the report meets quality standards.")
    score:                   float     = Field(description="Quality score 0-10.", ge=0, le=10)
    strengths:               list[str] = Field(description="What the report does well.")
    weaknesses:              list[str] = Field(description="Areas that fall short.")
    improvement_suggestions: list[str] = Field(description="Actionable suggestions.")

In [ ]:
if TEST_MODE:
    # Minimal token limits -- enough for the pipeline to run but not produce long output.
    # Use this to verify all agents and tools connect before the real recording.
    DEFAULT_SETTINGS  = ModelSettings(temperature=0.3, max_tokens=256,  top_p=0.95)
    SEARCH_SETTINGS   = ModelSettings(temperature=0.1, max_tokens=256,  top_p=0.90)
    CREATIVE_SETTINGS = ModelSettings(temperature=0.6, max_tokens=512,  top_p=0.95)
else:
    DEFAULT_SETTINGS  = ModelSettings(temperature=0.3, max_tokens=4096, top_p=0.95)
    SEARCH_SETTINGS   = ModelSettings(temperature=0.1, max_tokens=2048, top_p=0.90)
    CREATIVE_SETTINGS = ModelSettings(temperature=0.6, max_tokens=4096, top_p=0.95)

In [ ]:
@function_tool
def web_search(query: str) -> str:
    """Search the web using DuckDuckGo. Returns top 5 results with title, snippet and URL.
    Used instead of WebSearchTool so the pipeline works with any LLM provider."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return f"No results found for: {query}"
        return "\n\n---\n\n".join(
            f"{r.get('title', '')}\n{r.get('body', '')}\nSource: {r.get('href', '')}"
            for r in results
        )
    except Exception as exc:
        return f"Search error for '{query}': {exc}"


@function_tool
def send_report_email(subject: str, html_body: str) -> str:
    """Send the research report as a formatted HTML email via SendGrid."""
    api_key   = os.environ.get("SENDGRID_API_KEY")
    from_addr = os.environ.get("SENDGRID_FROM_EMAIL", "martin@markethacks.co.ke")
    to_addr   = os.environ.get("SENDGRID_TO_EMAIL",   "myskill254@gmail.com")
    if not api_key:
        return "error: SENDGRID_API_KEY not set"
    try:
        sg   = sendgrid.SendGridAPIClient(api_key=api_key)
        mail = Mail(Email(from_addr), To(to_addr), subject, Content("text/html", html_body))
        res  = sg.client.mail.send.post(request_body=mail.get())
        return f"sent (status {res.status_code})"
    except Exception as exc:
        return f"error: {exc}"

In [ ]:
clarifier_agent = Agent(
    name="ClarifierAgent",
    instructions="Given a research query, generate exactly three clarifying questions that sharpen the scope, and provide a one-sentence summary.",
    model=DEFAULT_MODEL, model_settings=DEFAULT_SETTINGS, output_type=ClarifyingQuestions,
)

planner_agent = Agent(
    name="PlannerAgent",
    instructions="Given a query and clarifications, produce 5-8 prioritised web search terms (priority 1=critical, 2=important, 3=nice-to-have).",
    model=DEFAULT_MODEL, model_settings=DEFAULT_SETTINGS, output_type=WebSearchPlan,
)

search_agent = Agent(
    name="SearchAgent",
    instructions="Given a search term, use web_search and produce a 2-3 paragraph summary (max 300 words). Note-taking style, no filler. Always call the tool.",
    model=DEFAULT_MODEL, model_settings=SEARCH_SETTINGS, tools=[web_search],
)

sufficiency_agent = Agent(
    name="SufficiencyAgent",
    instructions="Given the query and accumulated search results, decide whether evidence is sufficient for a thorough report. Suggest up to 3 additional queries if gaps remain.",
    model=DEFAULT_MODEL, model_settings=DEFAULT_SETTINGS, output_type=ResearchSufficiencyCheck,
)

writer_agent = Agent(
    name="WriterAgent",
    instructions="Given the query and search results, produce a detailed Markdown report (1000-2000 words): executive summary, background, key findings, analysis, conclusions, follow-up questions. Cite sources inline.",
    model=DEFAULT_MODEL, model_settings=CREATIVE_SETTINGS, output_type=ReportData,
)

evaluator_agent = Agent(
    name="EvaluatorAgent",
    instructions="Score the report 0-10. Approve (score >= 7) only if it fully addresses the query, is factually grounded, well structured, and at least 800 words. For lower scores provide actionable improvement suggestions.",
    model=DEFAULT_MODEL, model_settings=DEFAULT_SETTINGS, output_type=ReportEvaluation,
)

email_agent = Agent(
    name="EmailAgent",
    instructions="Convert the Markdown research report to clean HTML and send it as a single email with a descriptive subject line using the send_report_email tool.",
    model=DEFAULT_MODEL, model_settings=DEFAULT_SETTINGS, tools=[send_report_email],
)

In [ ]:
class _SafetyResult(BaseModel):
    is_safe: bool
    reason:  str

_safety_classifier = Agent(
    name="SafetyClassifier",
    instructions=(
        "Decide whether the research query is safe. Flag UNSAFE for: instructions to cause harm, "
        "illegal activity, synthesis of dangerous substances, or harassment of private individuals. "
        "Legitimate research on sensitive topics is SAFE."
    ),
    model=DEFAULT_MODEL,
    model_settings=ModelSettings(temperature=0.0, max_tokens=128),
    output_type=_SafetyResult,
)

@input_guardrail
async def query_safety_guardrail(ctx: RunContextWrapper, agent: Agent, input: str) -> GuardrailFunctionOutput:
    """
    Input guardrail: LLM-based safety classifier.
    Blocks harmful or illegal research queries before any tool calls run.
    Fails open (allows request) if the classifier itself errors.
    """
    try:
        result = await Runner.run(_safety_classifier, input, context=ctx.context)
        check: _SafetyResult = result.final_output
        return GuardrailFunctionOutput(output_info=check, tripwire_triggered=not check.is_safe)
    except Exception:
        return GuardrailFunctionOutput(
            output_info=_SafetyResult(is_safe=True, reason="classifier unavailable"),
            tripwire_triggered=False,
        )

In [ ]:
# In test mode both retry loops are disabled -- the pipeline runs exactly once
# through each stage: plan -> search -> sufficiency -> write -> evaluate -> email.
MAX_SEARCH_RETRIES = 0 if TEST_MODE else 2
MAX_REPORT_RETRIES = 0 if TEST_MODE else 1

@function_tool
async def run_parallel_searches(queries_json: str) -> str:
    """Execute web searches in parallel. Pass a JSON array of query strings."""
    try:
        queries: list[str] = json.loads(queries_json)
    except json.JSONDecodeError:
        queries = [q.strip() for q in queries_json.split("\n") if q.strip()]

    async def _one(q: str) -> str:
        try:
            r = await Runner.run(search_agent, f"Search term: {q}")
            return f"### {q}\n{r.final_output}\n"
        except Exception as exc:
            return f"### {q}\nFailed: {exc}\n"

    results = await asyncio.gather(*[_one(q) for q in queries[:10]])
    return "\n---\n".join(results)


_INSTRUCTIONS = f"""
You are a Deep Research Orchestrator. The user query and clarifying scope questions are
already provided in your input -- do not call any clarification tool.

Tools:
  - plan_searches         : turn the query + clarifications into search terms
  - run_parallel_searches : run all searches concurrently (pass a JSON array of strings)
  - check_sufficiency     : decide if evidence is enough; may suggest more queries
  - write_report          : synthesise evidence into a Markdown report
  - evaluate_report       : score the report 0-10; approve or request revision

Workflow:
STEP 1  Call plan_searches with the full query and scope context.
STEP 2  Call run_parallel_searches with the query strings from the plan (JSON array).
STEP 3  Call check_sufficiency. If insufficient and retries < {MAX_SEARCH_RETRIES}, re-plan and re-search.
STEP 4  Call write_report with ALL accumulated search results.
STEP 5  Call evaluate_report. If not approved and retries < {MAX_REPORT_RETRIES}, call write_report again.
STEP 6  Hand off to the email agent.

Output a brief status line at each step.
"""

orchestrator = Agent(
    name="ResearchOrchestrator",
    instructions=_INSTRUCTIONS,
    model=DEFAULT_MODEL,
    model_settings=ModelSettings(temperature=0.2, max_tokens=4096, top_p=0.95),
    tools=[
        # clarify_query is intentionally excluded -- clarification runs before the
        # orchestrator starts (Stage 1 of run_research) and is passed in via full_input.
        planner_agent.as_tool(    "plan_searches",       "Produce a prioritised list of search queries."),
        run_parallel_searches,
        sufficiency_agent.as_tool("check_sufficiency",   "Decide whether the research is sufficient."),
        writer_agent.as_tool(     "write_report",         "Write a detailed Markdown report."),
        evaluator_agent.as_tool(  "evaluate_report",      "Score the report and approve or request revision."),
    ],
    handoffs=[email_agent],
    input_guardrails=[query_safety_guardrail],
)

In [ ]:
async def run_research(query: str):
    """
    Full pipeline. Yields (status, report) tuples for the Gradio UI.
    Output guardrail: word-count check on the extracted report text.
    """
    if not query.strip():
        yield "Please enter a research query.", ""
        return

    status, report = "", ""

    # Stage 1: auto-clarify scope (non-blocking)
    status += "Analysing research scope...\n"
    yield status, report
    clarifications = ""
    try:
        cr = await Runner.run(clarifier_agent, query.strip())
        qs      = cr.final_output.questions[:3]
        summary = getattr(cr.final_output, "context_summary", "")
        status += f"{summary}\n\nResearch scope:\n" + "\n".join(f"  - {q}" for q in qs) + "\n\n"
        yield status, report
        clarifications = "Scope questions:\n" + "\n".join(f"- {q}" for q in qs)
    except Exception as exc:
        status += f"Warning: Could not clarify scope: {exc}\n\n"
        yield status, report

    # Stage 2: agentic research pipeline (streaming)
    full_input = f"Research query: {query}"
    if clarifications:
        full_input += f"\n\nUser clarifications:\n{clarifications}"

    with trace("Deep Research", trace_id=gen_trace_id()):
        status += "Starting research...\n"
        yield status, report
        try:
            stream = Runner.run_streamed(orchestrator, full_input, max_turns=20)
            async for event in stream.stream_events():
                etype = getattr(event, "type", "")
                if etype == "agent_updated_stream_event":
                    status += f"Agent: {event.new_agent.name}\n"
                    yield status, report
                elif etype == "run_item_stream_event":
                    item  = getattr(event, "item", None)
                    itype = type(item).__name__ if item else ""
                    if "ToolCall" in itype and "Output" not in itype:
                        raw  = getattr(item, "raw_item", {})
                        name = (raw.get("name") or raw.get("function", {}).get("name", "tool")) if isinstance(raw, dict) else getattr(raw, "name", "tool")
                        status += f"Calling: {name}\n"
                        yield status, report
                    elif "Handoff" in itype:
                        status += "Handing off to email agent...\n"
                        yield status, report

            # The orchestrator has no output_type so stream.final_output is just its
            # last sentence. The actual report is inside the write_report tool output.
            # Scan all run items for the write_report tool result first.
            report_text = ""
            for item in stream.new_items:
                raw = getattr(item, "raw_item", None)
                if isinstance(raw, dict) and raw.get("type") == "function_call_output":
                    try:
                        data = json.loads(raw.get("output", "{}"))
                        if "markdown_report" in data:
                            report_text = data["markdown_report"]
                            break
                    except (json.JSONDecodeError, TypeError, AttributeError):
                        pass

            # Fall back to final_output if the tool scan found nothing
            if not report_text:
                final = stream.final_output
                report_text = (
                    final.markdown_report if hasattr(final, "markdown_report") else str(final or "")
                )

            if report_text:
                report = report_text
                wc = len(report.split())
                # Output guardrail: minimum word-count check on the extracted report
                status += f"Warning: Report too short ({wc} words). Try again.\n" if wc < 200 else f"Report ready ({wc} words).\n"
                yield status, report
            else:
                status += "Warning: No report found in output. Try running again.\n"
                yield status, report

        except InputGuardrailTripwireTriggered as exc:
            status += f"Blocked: Query did not pass safety check -- {exc}\n"
            yield status, report
        except Exception as exc:
            status += f"Error during research: {exc}\n"
            yield status, report

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(primary_hue="sky"), title="Deep Research Agent") as demo:
    gr.Markdown("""
# Deep Research Agent
Enter a topic and click **Run Research**. The agent clarifies scope, runs parallel
web searches, checks sufficiency, writes and evaluates a report, then emails the result.

Pipeline: Clarify -> Plan -> Search (parallel) -> Sufficiency check -> Write -> Evaluate -> Email
""")
    with gr.Row():
        with gr.Column(scale=1):
            query_box    = gr.Textbox(label="What would you like to research?", placeholder="e.g. What are the latest advances in fusion energy?", lines=4)
            research_btn = gr.Button("Run Research", variant="primary", size="lg")
            gr.Markdown("_Press Enter or click Run Research. Everything runs automatically._")
        with gr.Column(scale=2):
            with gr.Tab("Live status"):
                status_box = gr.Textbox(label="", lines=20, interactive=False, show_copy_button=True,
                                        placeholder="Status updates will appear here as the agent works...")
            with gr.Tab("Final report"):
                report_box = gr.Markdown("The report will appear here when ready.")

    research_btn.click(fn=run_research, inputs=[query_box], outputs=[status_box, report_box])
    query_box.submit(  fn=run_research, inputs=[query_box], outputs=[status_box, report_box])

demo.launch(inbrowser=True)